# Data and model exploration
Data source [link](https://www.kaggle.com/datasets/andrewmvd/spotify-playlists)

## Big questions
- How can we validate our model is working?
  - Try to predict if recommendations show on other user's playlist, and that rate
  - Have classmates participate, and have them take 10 or so recommendations and tell us what % they like

In [113]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [114]:
# read in data
df = pd.read_csv('../../data/spotify_dataset.csv', on_bad_lines='skip')
df.columns=["user","artist","track","playlist"]
df.head()

,user,artist,track,playlist
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010


In [115]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12891680 entries, 0 to 12891679
Data columns (total 4 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   user      object
 1   artist    object
 2   track     object
 3   playlist  object
dtypes: object(4)
memory usage: 393.4+ MB


In [116]:
df.describe()

,user,artist,track,playlist
count,12891680,12858112,12891595,12890434
unique,15918,289821,2032044,157504
top,4398de6902abde3351347b048fcdc287,Daft Punk,Intro,Starred
freq,295275,36086,6676,1337085


# Data exploration

## Questions

- How many artists do we have?
  - 289,821
- How many users do we have?
  - 15,918
- How many playlists do users have?
  - 15, on average
- What are the top 5 most popular artists?
  - Coldplay
  - Daft Punk
  - Rihanna
  - David Guetta
  - Calvin Harris
- What are the top 5 most popular songs?
  - m83 midnightcity
  - daftpunk getlucky-radioedit
  - imaginedragons radioactive
  - ofmonstersandmen littletalks
  - avicii wakemeup

In [117]:
# number of artists
num_artists = df['artist'].nunique()
num_artists

289821

In [118]:
# number of users
num_users = df['user'].nunique()
num_users

15918

In [119]:
# avg number of playlists per user
playlist_user = df[['user', 'playlist']].drop_duplicates()
playlist_user.head()

,user,playlist
0,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010
67,9cc0cfd4d7d7885102480dd99e7a90d6,IOW 2012
104,07f0fc3be95dcd878966b1f9572ff670,2080
114,07f0fc3be95dcd878966b1f9572ff670,C418
148,07f0fc3be95dcd878966b1f9572ff670,Chill out


In [120]:
# user playlist counts
playlist_user_counts = playlist_user.groupby('user')['playlist'].count()
playlist_user_counts.mean()

14.562382208820203

# Data preprocessing

In [121]:
# create combined artist/track feature
def clean_text(text):
    """
    Drops spaces and lower-cases text
    """
    return str(text).lower().replace(' ', '')

df['artist_track'] = (df['artist'].apply(clean_text) + ' ' + 
                      df['track'].apply(clean_text))
df.head()

,user,artist,track,playlist,artist_track
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello (theangelswannawearmy)redshoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions (what'ssofunny'bo...
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage 7yearstoolate
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions accidentswillhappen
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello alison


In [122]:
# run user counts for each artist_track and each artist - can be used to measure song popularity
track_user_count = df.groupby('artist_track')['user'].count()
track_user_count = pd.DataFrame(track_user_count).reset_index()
track_user_count.columns = ['artist_track', 'count']
track_user_count.sort_values('count', ascending=False).head()

,artist_track,count
1462170,m83 midnightcity,2609
517563,daftpunk getlucky-radioedit,2341
1049832,imaginedragons radioactive,2336
1764670,ofmonstersandmen littletalks,2263
181773,avicii wakemeup,2242


In [123]:
# find most popular artists (count of users with artist in playlist)
artist_user_count = df[['user', 'artist']].drop_duplicates() \
    .groupby('artist')['user'].count()
artist_user_count = pd.DataFrame(artist_user_count).reset_index()
artist_user_count.columns = ['artist', 'count']
artist_user_count.sort_values('count', ascending=False).head()

,artist,count
50551,Coldplay,4645
59037,Daft Punk,4631
210322,Rihanna,4092
63007,David Guetta,3861
39657,Calvin Harris,3732


In [124]:
# create artist_popularity and track_popularity features
# artist popularity
artist_user_count['artist_popularity'] = artist_user_count['count'] / num_users
artist_user_count = artist_user_count[['artist', 'artist_popularity']]
artist_user_count.sort_values('artist_popularity', ascending=False).head()

,artist,artist_popularity
50551,Coldplay,0.291808
59037,Daft Punk,0.290929
210322,Rihanna,0.257067
63007,David Guetta,0.242556
39657,Calvin Harris,0.234452


In [125]:
# track popularity
track_user_count['track_popularity'] = track_user_count['count'] / num_users
track_user_count = track_user_count[['artist_track', 'track_popularity']]
track_user_count.sort_values('track_popularity', ascending=False).head()

,artist_track,track_popularity
1462170,m83 midnightcity,0.163903
517563,daftpunk getlucky-radioedit,0.147066
1049832,imaginedragons radioactive,0.146752
1764670,ofmonstersandmen littletalks,0.142166
181773,avicii wakemeup,0.140847


In [126]:
# merge both with main dataset
print(len(df))
df = df.merge(track_user_count, how='left', on='artist_track')
df = df.merge(artist_user_count, how='left', on='artist')
print(len(df))
df.head()

12891680
12891680


,user,artist,track,playlist,artist_track,track_popularity,artist_popularity
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello (theangelswannawearmy)redshoes,0.004837,0.041965
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions (what'ssofunny'bo...,0.005403,0.029212
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage 7yearstoolate,0.000063,0.000063
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions accidentswillhappen,0.004774,0.029212
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello alison,0.013632,0.041965


## Model research links

- https://365datascience.com/tutorials/how-to-build-recommendation-system-in-python/


## Initial modeling steps

### MVP model
- combine artist and track to create unique ID for each song
  - vectorize track/artist names with count vectorizer (lengths are similar)

### User similarity
- Look into collaborative filtering, factoring in user IDs
  
### Exploratory
- extract information from playlist names
  - vectorize playlist names to get same "feeling"

In [127]:
# MVP model (only factors in artist-track info)
df_mvp = df[['artist_track']].copy().drop_duplicates()
# sample a smaller amount of data to avoid kernel crash
df_mvp = df_mvp.sample(n=15000, replace=False, random_state=42)
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df_mvp['artist_track'])
similarities = cosine_similarity(vectorized)

In [128]:
similarities = pd.DataFrame(similarities, 
                            columns=df_mvp['artist_track'], 
                            index=df_mvp['artist_track']).reset_index()
similarities.head()

artist_track,artist_track,rickbraun cadillacslim,robertwells music,ninorota ilpellegrinaggio,whitneyhouston it'snotrightbutit'sokay-club69clubmix,2unlimited unlimitedmegajam,ericdolphy softlyasinamorningsunrise-live,ghostbeach miracle(gigameshremix),"chetbaker,stangetz 3+1=5",bryanferry let'ssticktogether,...,prissyclerks blast-offgirls,johnbarry&hisorchestra beatgirl(maintitle),peetahmorgan byebye,bossfight underskerimörkagränder,becalmhoncho whatwehavemade,greenpointorchestra 6-utra,peggyleewithdavebarbourandhisorchestra i'mgonnawashthatmanrightoutofmyhair,robertschumann carnavalop.9-06,ogmaco beenthuggin,2pac theydon'tgiveafuckaboutus
0,rickbraun cadillacslim,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,robertwells music,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ninorota ilpellegrinaggio,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,whitneyhouston it'snotrightbutit'sokay-club69c...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2unlimited unlimitedmegajam,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [129]:
input = "2pac theydon'tgiveafuckaboutus"
recommendations = pd.DataFrame(similarities.nlargest(11, input)['artist_track'])
recommendations = recommendations[recommendations['artist_track']!=input]
print(recommendations)

                                            artist_track
2938                                     2pac somuchpain
5016                                   2pac whatzyaphone
9444                         2pac wondawhytheycallubytch
9979                             2pac shortywannabeathug
11112               2pac/snoopdogg 2ofamerikazmostwanted
6200                               2pac pac'slife(remix)
7817                      paulwall theydon'tknow(ft.mike
7026             earth,wind&fire theydon'tsee-remastered
11394  childishgambino theydon'tlikeme(ft.chancethera...
0                                 rickbraun cadillacslim


## MVP model notes

- only factors in name similarity (same artist, titles with same words)
- while artist matches could be accurate, they are also obvious